# Gold feature engineering and leakage audit

## Objective
Create the model-ready direct multi-horizon dataset, document every predictor and preprocessing decision, and prove that observed-demand features stop at the declared forecast boundary.

Each row represents one SKU-store, one origin, and one horizon from 1 to 28. Demand history is anchored at the origin; target-date calendar, SNAP, event, and planned-price fields are allowed because they are assumed known before ordering. This avoids recursive forecasting and prevents horizons 2 to 28 from consuming realized demand from earlier forecast days.


In [ ]:
profile = "dev"
run_id = "notebook-gold"
force = False
execute_stage = False
seed = 42


## Dataset design

Training origins are spaced every 28 days over the configured history window, with all official backtest origins added explicitly. Every origin is crossed with horizons 1 to 28 and joined to its realized target. The final forecast table uses the same feature contract but joins future calendar and weekly prices instead of future demand.

A conventional row-per-date supervised table was rejected because using `lag_1` recursively would require unknown sales from within the 28-day forecast window. The direct design makes forecast-time availability explicit at the cost of a larger table.


In [ ]:
from IPython.display import display
import pandas as pd
from pyspark.sql import functions as F

from retail_forecasting.config import load_config
from retail_forecasting.data.gold import run_gold
from retail_forecasting.data.spark import get_spark, table_path

config = load_config(profile)
execute_stage_enabled = str(execute_stage).strip().lower() in {"1", "true", "yes"}
stage_result = run_gold(config, run_id) if execute_stage_enabled else {"status": "reusing existing Gold tables"}
stage_result


In [ ]:
spark = get_spark(config, "notebook-gold-analysis")
features = spark.read.format("delta").load(str(table_path(config, "gold", "training_features"))).cache()
future = spark.read.format("delta").load(str(table_path(config, "gold", "forecast_features"))).cache()
daily = spark.read.format("delta").load(str(table_path(config, "silver", "sales_daily"))).cache()


## Shape and temporal coverage
The following audit confirms the number of origins, horizons, series, and target dates rather than assuming the feature build produced a complete Cartesian design.


In [ ]:
shape = features.agg(
    F.count("*").alias("rows"),
    F.countDistinct("series_id").alias("series"),
    F.countDistinct("origin_day").alias("origins"),
    F.countDistinct("horizon").alias("horizons"),
    F.min("origin_day").alias("first_origin"),
    F.max("origin_day").alias("last_origin"),
    F.min("target_date").alias("first_target"),
    F.max("target_date").alias("last_target"),
).toPandas()
future_shape = future.agg(
    F.count("*").alias("rows"),
    F.countDistinct("series_id").alias("series"),
    F.countDistinct("horizon").alias("horizons"),
    F.min("target_date").alias("first_target"),
    F.max("target_date").alias("last_target"),
).toPandas()
display(pd.concat({"training": shape, "forecast": future_shape}))


## Variable catalogue

Observed-demand variables are computed once at the origin and reused for every horizon. Known-future variables vary with the target day. Identifiers allow global models to learn shared and segment-specific behavior.


In [ ]:
catalogue = []
for lag in config.features.lags:
    catalogue.append((f"lag_{lag}", "demand history", f"Units {lag} row(s) before the origin row", "origin"))
for window in config.features.rolling_windows:
    catalogue.extend([
        (f"rolling_mean_{window}", "demand level", f"Mean units over the preceding {window} rows", "origin"),
        (f"rolling_std_{window}", "volatility", f"Population standard deviation over the preceding {window} rows", "origin"),
        (f"nonzero_rate_{window}", "intermittency", f"Share of positive-demand days over the preceding {window} rows", "origin"),
    ])
catalogue.extend([
    ("days_since_last_sale", "lifecycle/intermittency", "Days from origin to the latest earlier positive-demand day", "origin"),
    ("short_long_trend", "trend", "rolling_mean_7 / max(rolling_mean_28, 0.1)", "origin"),
    ("horizon", "forecast geometry", "Integer from 1 to 28", "known future"),
    ("target_wday, target_month", "calendar", "Target weekday and month", "known future"),
    ("target_event_type", "event", "Primary M5 event type on target day", "known future"),
    ("target_snap_CA/TX/WI", "programme", "State SNAP eligibility on target day", "known future"),
    ("target_sell_price", "price", "Planned weekly item-store price for target day", "known future"),
    ("price_missing", "price quality", "One when planned price is unavailable", "known future"),
    ("item/dept/category/store/state", "retail identity", "Global-model categorical identifiers", "static"),
])
feature_catalogue = pd.DataFrame(catalogue, columns=["variable", "family", "definition", "availability"])
display(feature_catalogue)


## Encoding, missing values, and normalization

| Candidate | Categorical handling | Numeric handling | Scaling | Target/loss |
| --- | --- | --- | --- | --- |
| Seasonal naive / moving average | Not applicable | Raw demand units | None | Deterministic point forecast plus residual quantiles |
| LightGBM / XGBoost | `OrdinalEncoder`; unseen and missing categories become `-1` | Coerce to numeric, remaining nulls become `0`, cast to `float32` | None; tree splits are scale-insensitive | Raw non-negative units with Tweedie objective |
| N-HiTS | Series identity handled by NeuralForecast panel | Price null becomes `0` only with `price_missing=1`; SNAP is collapsed to the series state | Robust scaler fitted inside the training workflow | Raw units with multi-quantile loss for q05/q50/q95 |

Gold fills missing target price with zero only after creating `price_missing`. Demand lags are left nullable when history is insufficient; the tree adapter converts those remaining nulls to zero. A global standard scaler was rejected because it would be unnecessary for trees and could leak distribution information if fitted before temporal splitting.


## Leakage and boundary audit
The current feature convention uses observations strictly before the origin row: `lag_1` equals demand on `origin_day - 1`, and rolling windows end on that same day. The check below proves that convention against Silver. Target-date fields are permitted only when operationally known in advance.


In [ ]:
date_mismatches = features.filter(
    F.datediff("target_date", "origin_date") != F.col("horizon")
).count()
lag_reference = daily.select(
    "series_id", F.col("day_num").alias("history_day"), F.col("units").alias("expected_lag_1")
)
lag_mismatches = (
    features.select("series_id", "origin_day", "lag_1").distinct()
    .join(
        lag_reference,
        (features.series_id == lag_reference.series_id)
        & (features.origin_day - 1 == lag_reference.history_day),
    )
    .filter(~F.col("lag_1").eqNullSafe(F.col("expected_lag_1")))
    .count()
)
audit = pd.DataFrame([
    {"check": "target_date - origin_date equals horizon", "failures": date_mismatches},
    {"check": "lag_1 equals Silver demand at origin_day - 1", "failures": lag_mismatches},
    {"check": "all horizons are between 1 and configured horizon", "failures": features.filter((F.col('horizon') < 1) | (F.col('horizon') > config.data.horizon)).count()},
])
display(audit)
assert audit["failures"].sum() == 0


### Boundary convention risk
The final contract calls `d_1941` the forecast origin but the current demand windows end at `d_1940`. This is conservative and leakage-safe, yet it discards the latest potentially available observation. It should be tested as a separate feature-contract version rather than changed silently, because changing it invalidates every existing backtest and registered model.


## Feature distributions and missingness
Quantiles expose skew and extreme values before model fitting. Null rates verify how often early origins lack sufficient lag history; price missingness is represented explicitly.


In [ ]:
review_columns = ["target", "lag_1", "lag_7", "lag_28", "lag_364", "rolling_mean_7", "rolling_mean_28", "nonzero_rate_28", "days_since_last_sale", "short_long_trend", "target_sell_price"]
distribution_rows = []
row_count = features.count()
for column in review_columns:
    values = features.select(
        F.avg(F.col(column).isNull().cast("double")).alias("null_rate"),
        F.expr(f"percentile_approx({column}, array(0.0, 0.5, 0.9, 0.99, 1.0), 10000)").alias("quantiles"),
    ).first()
    distribution_rows.append({"variable": column, **values.asDict()})
display(pd.DataFrame(distribution_rows))


## Decisions, alternatives, and remaining risks

- Direct multi-horizon rows are larger than a recursive dataset but preserve forecast-time causality.
- Ordinal identifiers are efficient for global trees but their numeric codes have no business distance; native categorical handling is a candidate experiment.
- The current price representation lacks relative discount, price momentum, and item-level reference price. These are important missing retail features.
- `days_since_last_sale` describes recency, but explicit age since first sale and assortment status are not yet modeled.
- Event and SNAP variables are associations, not causal effects.
- The high bottom-level WAPE must be segmented by demand regime and horizon before considering the model production-ready.


In [ ]:
features.unpersist()
future.unpersist()
daily.unpersist()
spark.stop()
